<a href="https://colab.research.google.com/github/chrismji/Datasci-and-Ai-in-Bus/blob/main/T4_Decision_Trees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Decision Trees

**Data Science and AI for Business**

*Instructor:  Chris Volinsky*

***

## Import Python Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import math
import matplotlib.pylab as plt
import seaborn as sns

%matplotlib inline
sns.set(style='ticks', palette='Set2')

from sklearn.model_selection import train_test_split

# Introducing Titanic Data

We will look at a clasic introductory dataset that contains demographic and traveling information for the Titanic passengers. The goal is to predict the survival of these passengers. We will only keep a few variables of interest and transform all of them to numeric variables. We will also drop some outliers.

Read in the Titanic Data [by clicking on this file](https://drive.google.com/uc?export=download&id=1mJgS2IOvkmJLrnoVNSNcIz4e07V3wFwA)

In [ ]:
# read in data into data frame "df" - keeping only the features we will use

df = pd.read_csv("titanic.csv")[["survived", "pclass", "sex", "age", "fare"]]
df.head()


In [ ]:
df.info()

## Exploring the data

Lets re-code some of our features so they match their type:

In [ ]:
# make pclass and sex a categorical
df["pclass"] = pd.Categorical(df["pclass"], ordered=True, categories=[1, 2, 3])
df["sex"] = pd.Categorical(df["sex"])

In [ ]:
df.describe()

Note that `.describe()` does not give you info about categoricals...use `.value_counts()`

In [ ]:
df.sex.value_counts()
df.pclass.value_counts(sort=False)

### Missing Values

It looks like there might be some missing values of some of these variables, becasue they have different counts.  Let's confirm

In [ ]:
na_count = df.isna().sum()

na_count

Wow.  There are a lot of missing values for the age variable.  We would want to check and see if the missing cases are different in any way.  Is the fact that they are missing correlated with our target? (aka: are they more or less likely to die?) Lets check.


In [ ]:
age_missing = df['age'].isnull()

x1 = df[age_missing]['survived'].mean()
x2 = df[~age_missing]['survived'].mean()

print("% Survived with Age Missing =",round(x1,3),
      "\n% Survived with Age Not Missing =",round(x2,3))

This is concerning.  The fact that Age is missing seems to be predictive of whether or not they survived (Why might this be?).   Therefore, we dont want to simply remove these cases.  

If we were doing a careful analysis, we would investigate to try and fill these values in, or we could build a predictive model.  But for now, let us do the simplest imputation, replacing the nulls with the median age value.  

In [ ]:
# fill NAs in age and fare feature with the median

df["age"] = df["age"].fillna(df["age"].median())
df["fare"] = df["fare"].fillna(df["fare"].median())

# there were two other rows that had NAs....lets remove those

df=df.dropna()

### Visualizing the Data

In [ ]:
# plot histograms for our numerics

df[['age','fare']].hist(bins=25, edgecolor='black', grid=False, figsize=(10,5), layout=(1,2))

Exchange rate \$1 (1918) = \$25 (2026)

In [ ]:
# what is the *base rate* of survival?
# since "survived" is a numeric, we can just take the mean()

perclived=df.survived.mean().round(3)

print("% of passengers who survived = ",perclived)

### Exploring Predictive Value of Features

We'd like to use data about the passengers to predict whether they will survive. Let's start by taking a look at how well some of the variables "split" the data according to our target.

In [ ]:
rows, cols = 2, 2
fig, axs = plt.subplots(ncols=cols, nrows=rows, figsize=(7*cols, 7*rows))
axs = axs.flatten()

# Plot continuous features (age and fare)
sns.boxplot(x='survived', y='fare', data=df, ax=axs[0],color='LightBlue')
axs[0].set_title('Survival by Fare')

sns.boxplot(x='survived', y='age', data=df, ax=axs[1])
axs[1].set_title('Survival by Age')

# Calculate and plot survival rate for 'pclass'
#pclass_survival = df.groupby('pclass', observed=False)['survived'].mean().reset_index()
sns.barplot(x='pclass', y='survived', data=df, ax=axs[2],errorbar=None,color='orange')
axs[2].set_title('Survival Rate by Pclass')

# # Calculate and plot survival rate for 'sex'
sns.barplot(x='sex', y='survived', data=df, ax=axs[3],errorbar=None,color='pink')
axs[3].set_title('Survival Rate by Sex')

plt.tight_layout()
plt.show()

Above we see boxplots that fare seems to be related to survival.  Now, think about how we would come up with a rule for a decision tree.  To help this,  let's plot the distribution of fare according to survival:

In [ ]:
sns.histplot(data=df[df.survived==0], x="fare", bins=range(0, 300, 10))
plt.show()
sns.histplot(data=df[df.survived==1], x="fare", bins=range(0, 300, 10))
plt.show()

On might conclude from this that people that paid less are less likely to survive. 😱

Where would you draw the line?  Here is another view of the same data, by binning the fare and looking at survival rate within the bins...




In [ ]:
fare_bins = range(0, 350, 10)
df['fare_binned'] = pd.cut(df['fare'], bins=fare_bins, right=False)

survival_by_fare_bin = df.groupby('fare_binned', observed=False)['survived'].agg(['mean', 'count']).reset_index()
survival_by_fare_bin.rename(columns={'mean': 'survived', 'count': 'sample_size'}, inplace=True)

plt.figure(figsize=(12, 7))
sns.barplot(x='fare_binned', y='survived', data=survival_by_fare_bin, color='orange',legend=False)

# Add sample size labels on top of each bar
for index, row in survival_by_fare_bin.iterrows():
    plt.text(index, row['survived'],f"{row['sample_size']}", color='black', ha="center", va='bottom')

df.drop('fare_binned', axis=1, inplace=True)

plt.title('Survival Rate by Fare Bin with Sample Size')
plt.xlabel('Fare Bins')
plt.ylabel('Survival Rate')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

**What do you think is a good split point for a decision tree rule?**. How effective is this threshold? Let's quantify it!

### Calculating Information Gain manually


**Entropy** ($H$) and **information gain** ($IG$) are useful tools for measuring the effectiveness of a split on the values of one variable for giving information on the value of another variable. Entropy measures how random data is, information gain is a measure of the reduction in randomness after performing a split.

We wrote Python functions to calculate entropy and information gain


In [ ]:
def entropy(target_column):
    # get the counts of each target value
    target_counts = target_column.value_counts().astype(float).values
    total = target_column.count()

    if total == 0:  # Handle cases where the column is empty
        return 0.0

    # compute probas
    probas = target_counts/total
    # Filter out zero probabilities to avoid log(0) which causes RuntimeWarning
    non_zero_probas = probas[probas > 0]
    # p_i * log_2 (p_i)
    entropy_components = non_zero_probas * np.log2(non_zero_probas)
    # return negative sum
    return - entropy_components.sum()

def information_gain(df, info_column, target_column, threshold):
    # split data
    data_above_thresh = df[df[info_column] > threshold]
    data_below_thresh = df[df[info_column] <= threshold]
    # get entropy
    H = entropy(df[target_column])
    entropy_above = entropy(data_above_thresh[target_column])
    entropy_below = entropy(data_below_thresh[target_column])
    # compute weighted average
    ct_above = data_above_thresh.shape[0]
    ct_below = data_below_thresh.shape[0]
    tot = float(df.shape[0])
    return H - entropy_above*ct_above/tot - entropy_below*ct_below/tot

Now that we have a way of calculating $H$ and $IG$, let's test our prior guess as a good split point as a split on fare allows us to predict people who will survive.

In [ ]:
threshold = FILL_IN_THRESHOLD_HERE
prior_entropy = entropy(df["survived"])
IG = information_gain(df, "fare", "survived", threshold)
print ("Using a threshold of %.2f gives an Information Gain of %.4f " % (threshold, IG))

Lets find the best one! Let's loop through all possible splits on fare and see what is the best!

In [ ]:
col = "fare"
maximum_ig = 0
maximum_threshold = 0

# cycle through all of the unique values of fare, and calculate IG for each one:

for thresh in df[col].unique():
  IG = information_gain(df, col, 'survived', thresh)
  if IG > maximum_ig:
    maximum_ig = IG
    maximum_threshold = thresh

# print out the best value

print ("The best split threshold for %s is %.2f - which provides a maximum IG of %.4f." % (col, maximum_threshold, maximum_ig ))


If you were to do this across ALL of our features, you would see that the BEST possible split occurs with the feature `sex` which results in an information gain of 0.206.

That exercise was to help you understand what is happening under the hood, but thankfully we have software to do it for us!  

In [ ]:
# @title
# EXTRA - to calculate entropy for `sex` using our function, we need to make it a numeric
# create a numeric version of sex and add it to df
df["sex_num"] = df["sex"].cat.codes

ig=information_gain(df, "sex_num", "survived", 0.5)
print("Information Gain is %.2f" % ig)
# remove sex_num from df
df.drop("sex_num", axis=1, inplace=True)

## Fitting a Tree Model

We found the single best split for this data.  A Decision Tree will continue to split the data recursively ("recursive partitioning"), building a tree-structured model that should provide better predictions.  

Let's use the `sklearn` implementation which includes some additional functionality.  Start by defining our X and our y, and splitting into training and test sets.

In [ ]:
# because we have categorical features, we need to create dummies before doing the analysis
# we also need to create dummies of these features for proper analysis
# note - dont need drop_first = True for a decision tree!

df = pd.get_dummies(df,columns=["pclass", "sex"])
df.head()

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Define (instantiate) the model
decision_tree = DecisionTreeClassifier(max_depth=None, criterion="entropy")


# create X and y...always a good idea to keep df intact

Xcols=df.columns
Xcols=Xcols.drop("survived")
X=df[Xcols]

y=df["survived"]



Now, lets build our training and test sets - choose your own `random_state`

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = ###)

In [ ]:
# Fit the model to the training set
tree_model=decision_tree.fit(X_train,y_train)

We now have a classification tree, let's visualize the results!

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(15,10))
plot_tree(tree_model, feature_names=Xcols)
plt.show()

Whoa!  This will almost surely overfit!  We will see later how to protect against his...

Lets make things easier by setting `max_depth=3` - then visualize the tree and explore it to see what is important

In [ ]:
decision_tree = DecisionTreeClassifier(max_depth=3, criterion="entropy")

tree_small = decision_tree.fit(X_train,y_train)

plt.figure(figsize=(15,10))
plot_tree(tree_small, feature_names=Xcols, filled=True, impurity=False, class_names=["died", "survived"])

plt.show()


### Finding Optimal Tree Depth

How good is this model?  Calculate the accuracy.  This is simply means the percent of cases that are correctly predicted in the leaf nodes of the tree.


In [ ]:
from sklearn import metrics

y_pred = decision_tree.predict(X_test)

accuracy = metrics.accuracy_score(y_pred, y_test)
print ( "Accuracy = ", round(accuracy,4))

Now we will use our training/test split to help us determine what is the best depth for the tree.  There are various parameters we can use to prevent overfitting.  The main parameter we will optimize is:

- `max_depth` -  how many layers to the tree

(Note that there are *many* different parameters we can optimize, for this and for other models.  Here is a sample for DecisionTreeClassifier:
- `min_samples_split` - the minimum number of samples needed at a node to split (default =2)
- `min_samples_leaf` - the minimum number of samples that can be in a leaf node
- `max_leaf_nodes` - limits the number of leaf nodes in the overall tree
- `min_impurity_decrease` - restricts the ability of the tree to create trivial splits)



Let's explore the depth of the tree, using `max_depth`

Using a for loop, fit trees to your training set of max_depth = [2,3,4,5,...,10] and report the accuracy on the test set for each.

What is YOUR optimal depth? (this might differ based on the `random_state` chosen)

Optional - change `random_state` and run again.

In [ ]:
# Do this!

# create an list of numbers to cycle through from 2 to 10
max_depth_vals = ##

for ### in ### :

  decision_tree = DecisionTreeClassifier(max_depth = ##)

  # Tell the model what is the "training" data and then train it

  decision_tree.fit(## , ##)

  y_pred = decision_tree.predict(##)

  accuracy = metrics.accuracy_score(## , ##)

  print("max_depth = ", ## ,"Accuracy:", ##)


Report what the optimal value of max_depth is, and what the accuracy_score is

Once we know the optimal depth, if this were a real-world problem, before we apply it to the real world, we would re-fit our model to the FULL data set.

(This is a contrived case because there is no actual real-world application to predicting Titanic deaths 😱 ).  But lets do it anyway.


In [ ]:
# now that we know the best max_depth, to find the best tree...apply it to the ENTIRE data set
# (aka use X and y instead of X_train and y_train)

opt_depth = ## your optimal tree size here ##

tree_final = DecisionTreeClassifier(max_depth=opt_depth, criterion="entropy")
tree_final.fit(X,y)

plt.figure(figsize=(25,20))
plot_tree(tree_final, feature_names=Xcols, class_names=["died", "survived"],filled=True, impurity=False)
plt.show()

### Predictions and Feature Importances

Use your final model `tree_final` to explore the probabilities of passengers in the test set, but concatenating the survival probability onto the features.


In [ ]:
survival_prob = tree_final.predict_proba(X_test)[:,1]

# Convert the numpy array to a pandas Series with the same index as X_test
surv_prob_series = pd.Series(survival_prob, name='survival_prob', index=X_test.index)

# Concatenate X_test and the new Series
pd.concat([X_test, surv_prob_series], axis=1).head(20)



Now, estimate the survival probability of Rose, a 17 year old female, who paid $125 for her first class ticket.  use `.predict_proba()` to extract the probability prediction

In [ ]:
## create a feature vector for Rose - it must have the same number of features as the X used to train the model
#
# Features are [[age, fare, pclass_1, pclass_2, pclass_3, sex_female, sex_male]]

rose_data=[[##,...,##]]

rose_df = pd.DataFrame(rose_data, columns=X.columns)

rose_pred = tree_final.predict_proba(rose_df)[0]

print("Probability of Survival is ", rose_pred[1].round(3))

Maybe do the same for Jack, a 22 year old male in third class, whose fare was 0?

You can use `.feature_importances_` applied to the tree model to get the importance (Total Information Gain) of each feature.  Find the importances of the four attributes and plot as a barplot.

In [ ]:
# find feature importances and plot

# print(predictor_cols, importances.round(4))

scores = tree_final.feature_importances_
names=X.columns


sorted_pairs = sorted(zip(scores, names), reverse=True)
sorted_scores, sorted_names = zip(*sorted_pairs)

# Create the bar plot
plt.figure(figsize=(10, 6))  # Optional: Adjust the figure size as needed
plt.bar(sorted_names, sorted_scores, color='skyblue')  # You can change the color

# Add title and labels
plt.title('Scores by Name')
plt.xlabel('Name')
plt.ylabel('Score')

# Display the plot
plt.xticks(rotation=45)  # Rotate names to prevent overlap
plt.show()